In [1]:
import io
import os
import boto3
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from dotenv import load_dotenv
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

load_dotenv(Path("../.env"))

s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("AISTOR_ENDPOINT"),
    aws_access_key_id=os.getenv("AISTOR_ACCESS_KEY"),
    aws_secret_access_key=os.getenv("AISTOR_SECRET_KEY")
)

In [2]:
BUCKET = "fish-classifier"

classes = [
    "Black Sea Sprat",
    "Gilt-Head Bream",
    "Hourse Mackerel",
    "Red Mullet",
    "Red Sea Bream",
    "Sea Bass",
    "Shrimp",
    "Striped Red Mullet",
    "Trout"
]

class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(classes))
model.load_state_dict(torch.load("../models/efficientnet_b0_fish.pth", map_location=device))
model = model.to(device)
model.eval()

print("Model loaded successfully")

Model loaded successfully


In [3]:
from sklearn.model_selection import train_test_split

class FishDataset(Dataset):
    def __init__(self, keys, class_to_idx, transform=None):
        self.keys = keys
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        key = self.keys[idx]
        class_name = key.split('/')[1]
        label = self.class_to_idx[class_name]

        response = s3.get_object(Bucket=BUCKET, Key=key)
        img = Image.open(io.BytesIO(response['Body'].read())).convert('RGB')

        if self.transform:
            img = self.transform(img)

        return img, label

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

all_keys = []
paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket=BUCKET, Prefix="raw/"):
    for obj in page.get('Contents', []):
        all_keys.append(obj['Key'])

_, temp_keys = train_test_split(all_keys, test_size=0.3, random_state=42)
_, test_keys = train_test_split(temp_keys, test_size=0.5, random_state=42)

test_dataset = FishDataset(test_keys, class_to_idx, transform=val_transforms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Test samples: {len(test_keys)}")

Test samples: 1350


In [4]:
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

accuracy = (all_preds == all_labels).mean()
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.9926


In [5]:
print(classification_report(all_labels, all_preds, target_names=classes))

                    precision    recall  f1-score   support

   Black Sea Sprat       0.98      1.00      0.99       153
   Gilt-Head Bream       0.99      0.99      0.99       141
   Hourse Mackerel       1.00      1.00      1.00       145
        Red Mullet       1.00      0.99      1.00       141
     Red Sea Bream       0.99      1.00      1.00       154
          Sea Bass       1.00      0.97      0.98       155
            Shrimp       1.00      0.99      1.00       162
Striped Red Mullet       0.98      0.99      0.99       151
             Trout       0.99      0.99      0.99       148

          accuracy                           0.99      1350
         macro avg       0.99      0.99      0.99      1350
      weighted avg       0.99      0.99      0.99      1350

